# DIMER Workshop: Comparing Tabular Foundation Models for Seven-Day Retail Demand-Band Classification

**Dataset:** `freshretailnet-band-h7` sample derived from FreshRetailNet-50K  
**Dataset source:** `examples/sample-data/freshretailnet-band-h7.zip` in `kurtvalcorza/mitra-classifier-pipeline`  
**Pinned dataset revision:** `469d91252f3583b38d08b5c4d90fef4848b93f24`  
**Pinned archive SHA-256:** `ad2d2a8729749bb055754e4867acfb048fc27816f4eb344961d962b59c0be6dd`  
**Model execution:** Mitra Classifier, TabDPT v1.2 Classifier, TabPFN-3 Classifier and TabICLv2 Classifier run locally in isolated Python 3.12 environments  
**Task:** classify observed retail sales seven days ahead into `low`, `mid`, or `high` demand bands  
**Primary model-selection metric:** balanced accuracy  
**Recommended audience:** college students with introductory Python and machine-learning experience

---

## What you will do

In this notebook, you will:

1. acquire and verify a fixed FreshRetailNet-derived classification artifact;
2. inspect its class balance and feature distributions without using the test labels for model selection;
3. build a baseline ladder from a majority predictor to boosted trees;
4. run four tabular foundation-model classifiers under one evaluation protocol;
5. compare accuracy, balanced accuracy, macro-F1, MCC, log loss, and multiclass ROC-AUC;
6. inspect confusion matrices and class-specific recall;
7. test whether stockout features contribute predictive information;
8. freeze the selected development runs before evaluating the test partition; and
9. export a reproducibility bundle with provenance, settings, results, and figures.

> **Important:** This is a training and demonstration exercise. The sample is a tutorial/sanity fixture, not benchmark evidence. The target is a band of **observed future sales**, which can be affected by stockouts and does not necessarily equal latent customer demand.


## Research-design context

The activity follows a controlled comparative pattern:

- one fixed three-class prediction task;
- one fixed temporal train/validation/test split;
- the same feature columns for every full-feature model;
- a common probability-aware metric suite; and
- an independent test partition used only after the development configuration is frozen.

### Main research question

> How do tabular foundation-model classifiers compare with conventional classification baselines when predicting whether observed sales seven days ahead fall into a low, mid, or high demand band?

### Why balanced accuracy is primary

The demand bands are defined from **training-only quantile edges**, but the later validation and test periods can have different class proportions. Ordinary accuracy can therefore reward a model for following the most common class in a shifted period. Balanced accuracy averages recall across classes and gives each band equal weight.

Supporting metrics still matter:

- **Accuracy:** overall fraction of correct labels.
- **Balanced accuracy:** average recall across classes.
- **Macro-F1:** F1 computed separately for each class, then averaged equally.
- **MCC:** a correlation-style summary of the multiclass confusion matrix.
- **Log loss:** quality of the full probability distribution; lower is better.
- **ROC-AUC (OvR macro):** class-ranking quality, averaged one-vs-rest.


## Learning objectives

By the end of the activity, you should be able to:

- distinguish classification from regression even when both use the same feature table;
- explain why the class bands must be constructed without validation/test leakage;
- interpret class balance, a confusion matrix, per-class recall, balanced accuracy, macro-F1, MCC, log loss, and ROC-AUC;
- distinguish **in-context conditioning** from **gradient fine-tuning**;
- compare foundation models with meaningful classical baselines;
- identify why probability columns must be aligned to the same class order before scoring;
- use validation data for model selection and reserve the test partition for one frozen evaluation; and
- state conclusions with appropriate scope and limitations.

### Short glossary

| Term | Meaning in this notebook |
|---|---|
| **Classification** | Predicting one categorical label from a fixed set of classes. |
| **Demand band** | The categorical target: `low`, `mid`, or `high` observed sales seven days ahead. |
| **In-context learning (ICL)** | A pretrained model conditions on labelled support rows without gradient-updating its weights. |
| **Fine-tuning** | Gradient updates adapt pretrained weights to the current dataset. |
| **Argmax rule** | Choose the class with the highest predicted probability. |
| **Class-order alignment** | Reorder probability columns so every model is scored against the same `low/mid/high` order. |
| **Freeze** | Record the selected successful validation runs before evaluating the test partition. |


## Dataset and experiment summary

The pinned archive contains:

| Partition | Rows | Purpose |
|---|---:|---|
| `train.csv` | 4,180 | Fit classical models or provide support/fine-tuning data |
| `val.csv` | 1,600 | Compare models and choose settings |
| `test.csv` | 1,600 | Final evaluation after the experiment is frozen |

Each row has 17 numeric input features and one categorical target.

The source builder first creates the same seven-day-ahead continuous target used by the regression sample, performs a **purged chronological split with seven-row embargoes**, then computes demand-band cut points from the **training partition only**. The validation and test targets are assigned using those fixed training-derived edges.

### Feature groups

- **Historical sales:** `lag_1`, `lag_7`, `lag_14`, rolling means, rolling standard deviation
- **Stockout:** `stockout_hours`, `roll_7_stockout`
- **Context:** discount, holiday/activity flags, precipitation, temperature, humidity, wind, day-of-week, month

### Foundation models

| Model | Default workshop mode | Optional mode | Weight licence note |
|---|---|---|---|
| Mitra Classifier | In-context | GPU fine-tuning | Apache-2.0 |
| TabDPT v1.2 Classifier | In-context | — | Apache-2.0 |
| TabPFN-3 Classifier | In-context | Private-worker fine-tuning is outside this notebook | TabPFN-3 licence v1.0; non-commercial weights |
| TabICLv2 Classifier | In-context | GPU fine-tuning | BSD-3-Clause |

The notebook records per-model failures explicitly. If all selected foundation models fail, it stops instead of presenting a baseline-only table as a multi-model comparison.


## Before you begin

### Requirements

- A current Google Colab or Jupyter runtime
- Internet access to download the pinned dataset, clone the four pinned pipeline repositories, and retrieve their immutable model checkpoints
- Several gigabytes of free runtime storage
- A GPU is recommended for the full four-model exercise and required for optional Mitra/TabICLv2 fine-tuning

The notebook kernel may use a newer Python release. Foundation models run in isolated **Python 3.12** environments created with `uv`, which provides a common compatible interpreter for all four pinned pipelines.

### Test-set rule

Do not inspect test labels while developing the models. The notebook keeps test-target summaries out of EDA and validation comparisons. The test cell requires a recorded freeze manifest.

### Licence boundary

TabPFN-3 weights are non-commercial. Their inclusion here is for testing, evaluation, learning, and internal benchmarking under the upstream licence; it does not grant production or commercial deployment rights.


# 0. Set up the workshop runtime

In [ ]:
# @title 0.1 Imports, host dependencies, and workspace
from __future__ import annotations

import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import pickle
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
import uuid
import zipfile
from pathlib import Path, PurePosixPath

# Install only a missing host-side dependency. Foundation-model stacks are isolated later.
if importlib.util.find_spec("lightgbm") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "lightgbm==4.6.0"],
        check=True,
    )

import lightgbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

WORKSHOP_REVISION = "1.0.0"
RANDOM_SEED = 42
PRIMARY_METRIC = "balanced_accuracy"
TARGET_COLUMN = "target"
SEMANTIC_CLASS_ORDER = ["low", "mid", "high"]

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
WORKSPACE_ROOT = ROOT / "dimer_classifier_workshop_v1"
CACHE_ROOT = WORKSPACE_ROOT / "cache"
REPOSITORY_ROOT = WORKSPACE_ROOT / "repositories"
ENVIRONMENT_ROOT = WORKSPACE_ROOT / "environments"
SESSION_ROOT = WORKSPACE_ROOT / "sessions" / uuid.uuid4().hex[:12]
DATA_ROOT = SESSION_ROOT / "data"
RUN_ROOT = SESSION_ROOT / "runs"
FIGURE_ROOT = SESSION_ROOT / "figures"
EXPORT_ROOT = SESSION_ROOT / "exports"
SUPPORT_ROOT = SESSION_ROOT / "support"

for directory in (
    WORKSPACE_ROOT,
    CACHE_ROOT,
    REPOSITORY_ROOT,
    ENVIRONMENT_ROOT,
    SESSION_ROOT,
    DATA_ROOT,
    RUN_ROOT,
    FIGURE_ROOT,
    EXPORT_ROOT,
    SUPPORT_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def canonical_json(value) -> str:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def fingerprint(value) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()

def write_json(path: str | Path, value) -> None:
    Path(path).write_text(
        json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )

def file_inventory(root: str | Path) -> dict[str, str]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(root)
    inventory = {}
    for path in sorted(root.rglob("*")):
        if path.is_symlink():
            raise ValueError(f"Artifact inventory refuses symbolic links: {path}")
        if path.is_file():
            inventory[path.relative_to(root).as_posix()] = sha256_file(path)
    return inventory

def verify_file_inventory(root: str | Path, expected: dict[str, str]) -> None:
    root = Path(root)
    observed = file_inventory(root)
    if observed != expected:
        missing = sorted(set(expected) - set(observed))
        extra = sorted(set(observed) - set(expected))
        changed = sorted(
            name for name in set(expected) & set(observed)
            if expected[name] != observed[name]
        )
        raise ValueError(
            "Frozen artifact inventory changed. "
            f"Missing={missing}; extra={extra}; changed={changed}"
        )

def class_order(values) -> list[str]:
    observed = {str(value) for value in values}
    ordered = [label for label in SEMANTIC_CLASS_ORDER if label in observed]
    ordered.extend(sorted(observed - set(ordered)))
    return ordered

def align_probabilities(raw, model_classes, classes) -> np.ndarray:
    array = np.asarray(raw, dtype=float)
    source = [str(value) for value in model_classes]
    target = [str(value) for value in classes]
    if array.ndim != 2 or array.shape[1] != len(source):
        raise ValueError("Probability array shape does not match the model class list.")
    mapping = {label: i for i, label in enumerate(source)}
    missing = [label for label in target if label not in mapping]
    if missing:
        raise ValueError(f"Model probabilities omit required classes: {missing}")
    aligned = array[:, [mapping[label] for label in target]]
    if not np.isfinite(aligned).all():
        raise ValueError("Probabilities contain non-finite values.")
    row_sums = aligned.sum(axis=1)
    if not np.allclose(row_sums, 1.0, rtol=1e-5, atol=1e-6):
        raise ValueError("Probability rows do not sum to approximately 1.")
    return aligned

def classification_metrics(y_true, y_pred, proba, classes) -> dict[str, float]:
    y = np.asarray([str(v) for v in y_true])
    pred = np.asarray([str(v) for v in y_pred])
    p = np.asarray(proba, dtype=float)
    labels = [str(v) for v in classes]
    if y.shape != pred.shape or len(y) == 0:
        raise ValueError("Targets and predictions must be non-empty and equally sized.")
    if p.shape != (len(y), len(labels)):
        raise ValueError("Probability matrix does not match rows/classes.")

    label_to_index = {label: index for index, label in enumerate(labels)}
    try:
        y_index = np.asarray([label_to_index[value] for value in y], dtype=int)
        pred_index = np.asarray([label_to_index[value] for value in pred], dtype=int)
    except KeyError as exc:
        raise ValueError(
            f"Unknown class label in targets/predictions: {exc.args[0]!r}"
        ) from exc

    numeric_labels = list(range(len(labels)))
    result = {
        "accuracy": float(accuracy_score(y_index, pred_index)),
        "balanced_accuracy": float(balanced_accuracy_score(y_index, pred_index)),
        "f1_macro": float(
            f1_score(
                y_index,
                pred_index,
                labels=numeric_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "mcc": float(matthews_corrcoef(y_index, pred_index)),
        "log_loss": float(log_loss(y_index, p, labels=numeric_labels)),
    }
    try:
        if len(labels) == 2:
            result["roc_auc_ovr_macro"] = float(
                roc_auc_score(y_index, p[:, 1])
            )
        else:
            y_binary = label_binarize(
                y_index,
                classes=np.arange(len(labels)),
            )
            result["roc_auc_ovr_macro"] = float(
                roc_auc_score(y_binary, p, average="macro")
            )
    except ValueError:
        result["roc_auc_ovr_macro"] = float("nan")
    return result

def prediction_frame(reference: pd.DataFrame, predictions, proba, classes) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "row_id": np.arange(len(reference)),
            "target": reference[TARGET_COLUMN].astype(str).to_numpy(),
            "prediction": np.asarray(predictions).astype(str),
        }
    )
    for index, label in enumerate(classes):
        out[f"probability_{label}"] = np.asarray(proba, dtype=float)[:, index]
    return out

SOFTWARE_VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "lightgbm": lightgbm.__version__,
}

print("Session:", SESSION_ROOT)
display(pd.Series(SOFTWARE_VERSIONS, name="version").to_frame())


# 1. Acquire the pinned FreshRetailNet demand-band sample

This workshop downloads one immutable repository artifact. There is no DIMER dataset fallback and no alternate sample fallback.

- Repository: `kurtvalcorza/mitra-classifier-pipeline`
- Revision: `469d91252f3583b38d08b5c4d90fef4848b93f24`
- Path: `examples/sample-data/freshretailnet-band-h7.zip`
- SHA-256: `ad2d2a8729749bb055754e4867acfb048fc27816f4eb344961d962b59c0be6dd`

A mismatch stops the notebook.


In [ ]:
# @title 1.1 Download, hash, and safely stage the archive
DATASET_REPOSITORY = "kurtvalcorza/mitra-classifier-pipeline"
DATASET_REPOSITORY_REVISION = "469d91252f3583b38d08b5c4d90fef4848b93f24"
DATASET_REPOSITORY_PATH = "examples/sample-data/freshretailnet-band-h7.zip"
EXPECTED_DATASET_SHA256 = "ad2d2a8729749bb055754e4867acfb048fc27816f4eb344961d962b59c0be6dd"
DATASET_URL = (
    "https://raw.githubusercontent.com/"
    f"{DATASET_REPOSITORY}/{DATASET_REPOSITORY_REVISION}/{DATASET_REPOSITORY_PATH}"
)

archive_path = DATA_ROOT / "freshretailnet-band-h7.zip"
print("Downloading:", DATASET_URL)
urllib.request.urlretrieve(DATASET_URL, archive_path)

if not zipfile.is_zipfile(archive_path):
    raise ValueError("Downloaded dataset is not a valid ZIP archive.")

DATASET_SHA256 = sha256_file(archive_path)
if DATASET_SHA256 != EXPECTED_DATASET_SHA256:
    raise ValueError(
        "Dataset digest mismatch. "
        f"Expected {EXPECTED_DATASET_SHA256}, observed {DATASET_SHA256}."
    )

required_members = {"train.csv", "val.csv", "test.csv"}
staged_paths = {}
extract_root = DATA_ROOT / "staged"
extract_root.mkdir(exist_ok=False)

with zipfile.ZipFile(archive_path) as zf:
    selected = {}
    expanded_bytes = 0
    for info in zf.infolist():
        name = info.filename
        posix = PurePosixPath(name)
        if "\\" in name or posix.is_absolute() or ".." in posix.parts:
            raise ValueError(f"Unsafe ZIP path: {name}")
        expanded_bytes += info.file_size
        if expanded_bytes > 512 * 1024 * 1024:
            raise ValueError("Dataset archive exceeds the 512 MiB expanded-size ceiling.")
        if not info.is_dir() and posix.name in required_members:
            if posix.name in selected:
                raise ValueError(f"Duplicate split member: {posix.name}")
            selected[posix.name] = info
    missing = required_members - set(selected)
    if missing:
        raise ValueError(f"Dataset is missing required split files: {sorted(missing)}")
    for member_name, info in selected.items():
        target = extract_root / member_name
        target.write_bytes(zf.read(info))
        staged_paths[member_name[:-4]] = target

DATASET_PROVENANCE = {
    "dataset_name": "freshretailnet-band-h7",
    "repository": DATASET_REPOSITORY,
    "repository_revision": DATASET_REPOSITORY_REVISION,
    "repository_path": DATASET_REPOSITORY_PATH,
    "source_url": DATASET_URL,
    "archive_bytes": archive_path.stat().st_size,
    "sha256": DATASET_SHA256,
    "sha256_verified": True,
    "split_sha256": {name: sha256_file(path) for name, path in staged_paths.items()},
}
display(pd.Series(DATASET_PROVENANCE, name="value").to_frame())


# 2. Load and validate the train, validation, and test contracts

In [ ]:
# @title 2.1 Validate rows, schema, numeric features, and class coverage
EXPECTED_ROWS = {"train": 4180, "val": 1600, "test": 1600}
EXPECTED_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "roll_7_mean",
    "roll_28_mean",
    "roll_7_std",
    "stockout_hours",
    "roll_7_stockout",
    "discount",
    "holiday_flag",
    "activity_flag",
    "precpt",
    "avg_temperature",
    "avg_humidity",
    "avg_wind_level",
    "dow",
    "month",
]

frames = {
    split: pd.read_csv(path, float_precision="round_trip")
    for split, path in staged_paths.items()
}

reference_columns = EXPECTED_FEATURES + [TARGET_COLUMN]
for split, frame in frames.items():
    if len(frame) != EXPECTED_ROWS[split]:
        raise ValueError(
            f"{split}: expected {EXPECTED_ROWS[split]} rows, observed {len(frame)}."
        )
    if list(frame.columns) != reference_columns:
        raise ValueError(
            f"{split}: schema mismatch.\n"
            f"Expected: {reference_columns}\nObserved: {list(frame.columns)}"
        )
    if frame.columns.duplicated().any():
        raise ValueError(f"{split}: duplicate columns are not allowed.")
    for column in EXPECTED_FEATURES:
        numeric = pd.to_numeric(frame[column], errors="raise")
        if not np.isfinite(numeric.to_numpy(dtype=float)).all():
            raise ValueError(f"{split}: feature {column!r} contains non-finite values.")
    if frame[TARGET_COLUMN].isna().any():
        raise ValueError(f"{split}: target contains missing values.")

CLASS_LABELS = class_order(frames["train"][TARGET_COLUMN].astype(str))
if CLASS_LABELS != SEMANTIC_CLASS_ORDER:
    raise ValueError(
        f"Expected semantic classes {SEMANTIC_CLASS_ORDER}, observed {CLASS_LABELS}."
    )

for split in ("val", "test"):
    observed_classes = set(frames[split][TARGET_COLUMN].astype(str))
    unseen = sorted(observed_classes - set(CLASS_LABELS))
    missing = sorted(set(CLASS_LABELS) - observed_classes)
    if unseen or missing:
        raise ValueError(
            f"{split}: class coverage mismatch; unseen={unseen}, missing={missing}"
        )

FEATURE_COLUMNS = list(EXPECTED_FEATURES)
X_train = frames["train"][FEATURE_COLUMNS]
y_train = frames["train"][TARGET_COLUMN].astype(str)
X_val = frames["val"][FEATURE_COLUMNS]
y_val = frames["val"][TARGET_COLUMN].astype(str)

DATASET_IDENTITY = {
    "archive_sha256": DATASET_SHA256,
    "split_sha256": DATASET_PROVENANCE["split_sha256"],
    "feature_columns": FEATURE_COLUMNS,
    "target_column": TARGET_COLUMN,
    "class_labels": CLASS_LABELS,
    "rows": EXPECTED_ROWS,
}
DATASET_FINGERPRINT = fingerprint(DATASET_IDENTITY)

print("Dataset fingerprint:", DATASET_FINGERPRINT)
display(
    pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "rows": [len(frames["train"]), len(frames["val"]), len(frames["test"])],
            "features": [len(FEATURE_COLUMNS)] * 3,
        }
    )
)
print("Classes:", CLASS_LABELS)


### Class balance

The next cell shows **training and validation** target proportions. The test target distribution stays hidden until after the freeze.

Because the band cut points came from the training partition, training classes are expected to be comparatively balanced. Later chronological periods may shift.


In [ ]:
# @title 2.2 Inspect development class balance only
development_class_balance = pd.concat(
    {
        "train": y_train.value_counts(normalize=True).reindex(CLASS_LABELS),
        "validation": y_val.value_counts(normalize=True).reindex(CLASS_LABELS),
    },
    axis=1,
).fillna(0.0)

display((development_class_balance * 100).round(2).rename_axis("class").rename(columns=lambda x: f"{x}_pct"))

fig, ax = plt.subplots(figsize=(8, 4))
development_class_balance.T.plot(kind="bar", ax=ax)
ax.set_title("Class proportions: development partitions")
ax.set_ylabel("Proportion")
ax.set_xlabel("")
plt.tight_layout()
class_balance_figure = FIGURE_ROOT / "development_class_balance.png"
plt.savefig(class_balance_figure, dpi=140)
plt.show()

print("Test rows:", len(frames["test"]))
print("Test target distribution intentionally not displayed before freeze.")


# 3. Exploratory data analysis

EDA should explain the table before it evaluates models. These views use the training and validation partitions only.

Focus on three questions:

1. Do the demand bands occupy similar proportions across development periods?
2. Which historical-sales features separate the bands?
3. Do stockout features shift across classes or periods?


In [ ]:
# @title 3.1 Development-only EDA
eda_train = frames["train"].copy()
eda_val = frames["val"].copy()

summary_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "roll_7_mean",
    "stockout_hours",
    "roll_7_stockout",
    "discount",
    "avg_temperature",
]

development_summary = pd.concat(
    {
        "train": eda_train[summary_columns].describe().T[["mean", "std", "min", "50%", "max"]],
        "validation": eda_val[summary_columns].describe().T[["mean", "std", "min", "50%", "max"]],
    },
    axis=1,
)
display(development_summary.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
for label in CLASS_LABELS:
    values = eda_train.loc[eda_train[TARGET_COLUMN] == label, "lag_7"]
    ax.hist(values, bins=30, alpha=0.45, label=label)
ax.set_title("Training distribution of lag_7 by future demand band")
ax.set_xlabel("lag_7")
ax.set_ylabel("Rows")
ax.legend()
plt.tight_layout()
lag_figure = FIGURE_ROOT / "lag7_by_class.png"
plt.savefig(lag_figure, dpi=140)
plt.show()

stockout_by_class = (
    eda_train.groupby(TARGET_COLUMN)[["stockout_hours", "roll_7_stockout"]]
    .agg(["mean", "median"])
    .reindex(CLASS_LABELS)
)
display(stockout_by_class.round(3))


# 4. Build a classical baseline ladder

A foundation model should not be compared only with a weak constant predictor.

This notebook uses:

1. **Training-prior majority:** always predict the most common training class; probabilities equal the training class shares.
2. **Lag-7 quantile-band heuristic:** classify `lag_7` using training-only tertile cut points as a simple temporal persistence heuristic.
3. **Multinomial logistic regression:** regularized linear decision surfaces after median imputation and standardization.
4. **Random Forest:** nonlinear tree ensemble.
5. **LightGBM:** gradient-boosted trees.

All models are evaluated on the same validation rows and the same class order.


In [ ]:
# @title 4.1 Fit and score the baseline ladder
BASELINE_ROOT = RUN_ROOT / "baselines"
BASELINE_ROOT.mkdir(exist_ok=True)

def register_baseline(
    key: str,
    model_name: str,
    predictions,
    probabilities,
    artifact,
    *,
    notes: str = "",
):
    proba = np.asarray(probabilities, dtype=float)
    pred = np.asarray(predictions).astype(str)
    metrics = classification_metrics(y_val, pred, proba, CLASS_LABELS)
    run_dir = BASELINE_ROOT / key
    run_dir.mkdir(exist_ok=False)
    with (run_dir / "artifact.pkl").open("wb") as handle:
        pickle.dump(artifact, handle)
    pred_frame = prediction_frame(frames["val"], pred, proba, CLASS_LABELS)
    pred_frame.to_csv(run_dir / "validation_predictions.csv", index=False)
    result = {
        "model_key": key,
        "model": model_name,
        "family": "Classical / heuristic",
        "condition": "validation",
        **metrics,
        "runtime_seconds": None,
        "device": "cpu",
        "licence": "n/a",
        "notes": notes,
        "artifact_sha256": sha256_file(run_dir / "artifact.pkl"),
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "feature_columns": FEATURE_COLUMNS,
    }
    write_json(run_dir / "result.json", result)
    return result, pred_frame

baseline_records = []
baseline_prediction_registry = {}

# 1) Training-prior majority baseline.
prior_series = y_train.value_counts(normalize=True).reindex(CLASS_LABELS).fillna(0.0)
majority_label = str(prior_series.idxmax())
majority_pred = np.repeat(majority_label, len(y_val))
majority_proba = np.tile(prior_series.to_numpy(dtype=float), (len(y_val), 1))
result, pred_frame = register_baseline(
    "training_prior_majority",
    "Training-prior majority",
    majority_pred,
    majority_proba,
    {
        "kind": "majority",
        "class_labels": CLASS_LABELS,
        "priors": prior_series.to_dict(),
        "majority_label": majority_label,
    },
)
baseline_records.append(result)
baseline_prediction_registry[result["model_key"]] = pred_frame

# 2) Domain-informed lag-7 heuristic using train-only quantiles.
lag_edges = X_train["lag_7"].quantile([1/3, 2/3]).to_numpy(dtype=float)
lag_ids = np.digitize(X_val["lag_7"].to_numpy(dtype=float), lag_edges)
lag_pred = np.asarray(CLASS_LABELS, dtype=object)[lag_ids]
lag_proba = np.full((len(lag_pred), len(CLASS_LABELS)), 1e-6, dtype=float)
lag_proba[np.arange(len(lag_pred)), lag_ids] = 1.0 - (len(CLASS_LABELS) - 1) * 1e-6
result, pred_frame = register_baseline(
    "lag7_quantile_band",
    "Lag-7 quantile-band heuristic",
    lag_pred,
    lag_proba,
    {
        "kind": "lag7_quantile",
        "feature": "lag_7",
        "edges": lag_edges.tolist(),
        "class_labels": CLASS_LABELS,
    },
    notes="Training-only lag_7 tertiles; a simple temporal persistence heuristic.",
)
baseline_records.append(result)
baseline_prediction_registry[result["model_key"]] = pred_frame

# 3) Logistic regression.
logistic = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)),
    ]
)
t0 = time.perf_counter()
logistic.fit(X_train, y_train)
elapsed = time.perf_counter() - t0
logistic_raw = logistic.predict_proba(X_val)
logistic_classes = logistic.named_steps["model"].classes_
logistic_proba = align_probabilities(logistic_raw, logistic_classes, CLASS_LABELS)
logistic_pred = np.asarray(CLASS_LABELS, dtype=object)[np.argmax(logistic_proba, axis=1)]
result, pred_frame = register_baseline(
    "logistic_regression",
    "Logistic Regression",
    logistic_pred,
    logistic_proba,
    {"kind": "sklearn", "model": logistic, "class_labels": CLASS_LABELS},
)
result["runtime_seconds"] = elapsed
write_json(BASELINE_ROOT / "logistic_regression" / "result.json", result)
baseline_records.append(result)
baseline_prediction_registry[result["model_key"]] = pred_frame

# 4) Random Forest.
random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
t0 = time.perf_counter()
random_forest.fit(X_train, y_train)
elapsed = time.perf_counter() - t0
rf_proba = align_probabilities(
    random_forest.predict_proba(X_val),
    random_forest.classes_,
    CLASS_LABELS,
)
rf_pred = np.asarray(CLASS_LABELS, dtype=object)[np.argmax(rf_proba, axis=1)]
result, pred_frame = register_baseline(
    "random_forest",
    "Random Forest",
    rf_pred,
    rf_proba,
    {"kind": "sklearn", "model": random_forest, "class_labels": CLASS_LABELS},
)
result["runtime_seconds"] = elapsed
write_json(BASELINE_ROOT / "random_forest" / "result.json", result)
baseline_records.append(result)
baseline_prediction_registry[result["model_key"]] = pred_frame

# 5) LightGBM.
lightgbm_model = LGBMClassifier(
    n_estimators=300,
    random_state=RANDOM_SEED,
    verbosity=-1,
)
t0 = time.perf_counter()
lightgbm_model.fit(X_train, y_train)
elapsed = time.perf_counter() - t0
lgbm_proba = align_probabilities(
    lightgbm_model.predict_proba(X_val),
    lightgbm_model.classes_,
    CLASS_LABELS,
)
lgbm_pred = np.asarray(CLASS_LABELS, dtype=object)[np.argmax(lgbm_proba, axis=1)]
result, pred_frame = register_baseline(
    "lightgbm",
    "LightGBM",
    lgbm_pred,
    lgbm_proba,
    {"kind": "sklearn", "model": lightgbm_model, "class_labels": CLASS_LABELS},
)
result["runtime_seconds"] = elapsed
write_json(BASELINE_ROOT / "lightgbm" / "result.json", result)
baseline_records.append(result)
baseline_prediction_registry[result["model_key"]] = pred_frame

baseline_validation_results = pd.DataFrame(baseline_records)
baseline_validation_results = baseline_validation_results.sort_values(
    PRIMARY_METRIC,
    ascending=False,
).reset_index(drop=True)

display(
    baseline_validation_results[
        [
            "model",
            "balanced_accuracy",
            "accuracy",
            "f1_macro",
            "mcc",
            "log_loss",
            "roc_auc_ovr_macro",
        ]
    ].round(4)
)


# 5. Run the tabular foundation-model classifiers

The four pipeline repositories are checked out at fixed commits and installed in separate Python 3.12 virtual environments.

Default conditions:

- **Mitra Classifier:** in-context
- **TabDPT v1.2 Classifier:** in-context
- **TabPFN-3 Classifier:** in-context
- **TabICLv2 Classifier:** in-context

Optional GPU conditions:

- Mitra fine-tuned
- TabICLv2 fine-tuned

Each model must emit a probability matrix that is explicitly reordered to the workshop class order `low → mid → high` before scoring.


In [ ]:
# @title 5.1 Select foundation-model conditions
RUN_MITRA_ICL = True # @param {type:"boolean"}
RUN_TABDPT_ICL = True # @param {type:"boolean"}
RUN_TABPFN3_ICL = True # @param {type:"boolean"}
RUN_TABICLV2_ICL = True # @param {type:"boolean"}

RUN_MITRA_FINETUNED = False # @param {type:"boolean"}
RUN_TABICLV2_FINETUNED = False # @param {type:"boolean"}

DEVICE_PREFERENCE = "auto" # @param ["auto", "cpu", "cuda"]
REBUILD_MODEL_ENVIRONMENTS = False # @param {type:"boolean"}
FORCE_MODEL_RERUN = False # @param {type:"boolean"}

MITRA_TIME_LIMIT_SECONDS = 600
MITRA_FINE_TUNE_STEPS = 50

TABDPT_N_ENSEMBLES = 4
TABDPT_CONTEXT_SIZE = 4180
TABDPT_BATCH_SIZE = 4096

TABPFN_N_ESTIMATORS = 4
TABICL_N_ESTIMATORS = 8
TABICL_FINE_TUNE_EPOCHS = 10
TABICL_FINE_TUNE_TIME_LIMIT = 600
TABICL_FINE_TUNE_PATIENCE = 3

MODEL_REPOSITORIES = {
    "mitra": {
        "repository": "kurtvalcorza/mitra-classifier-pipeline",
        "commit": "469d91252f3583b38d08b5c4d90fef4848b93f24",
        "import_name": "mitra_pipeline",
        "licence": "Apache-2.0",
    },
    "tabdpt": {
        "repository": "kurtvalcorza/tabdpt-classifier-pipeline",
        "commit": "1e35dc4738f015b254b1724fd4bc7ba9f49668af",
        "import_name": "tabdpt_classifier_pipeline",
        "licence": "Apache-2.0",
    },
    "tabpfn": {
        "repository": "kurtvalcorza/tabpfn-classifier-pipeline",
        "commit": "769fb7b47e01b1d10edd954dabc634b8034598c2",
        "import_name": "tabpfn_classifier_pipeline",
        "licence": "TabPFN-3 licence v1.0 — non-commercial weights",
    },
    "tabicl": {
        "repository": "kurtvalcorza/tabicl-classifier-pipeline",
        "commit": "7400516ac86e3b6e55f5134ad5e550f312948981",
        "import_name": "tabicl_classifier_pipeline",
        "licence": "BSD-3-Clause",
    },
}

CONDITION_SPECS = {
    "mitra_icl": {
        "environment": "mitra",
        "display_name": "Mitra Classifier",
        "condition": "in_context",
        "configuration": {
            "fine_tune": False,
            "time_limit_seconds": MITRA_TIME_LIMIT_SECONDS,
            "fine_tune_steps": 0,
        },
    },
    "mitra_finetuned": {
        "environment": "mitra",
        "display_name": "Mitra Classifier",
        "condition": "fine_tuned",
        "configuration": {
            "fine_tune": True,
            "time_limit_seconds": MITRA_TIME_LIMIT_SECONDS,
            "fine_tune_steps": MITRA_FINE_TUNE_STEPS,
        },
    },
    "tabdpt_icl": {
        "environment": "tabdpt",
        "display_name": "TabDPT v1.2 Classifier",
        "condition": "in_context",
        "configuration": {
            "n_ensembles": TABDPT_N_ENSEMBLES,
            "context_size": TABDPT_CONTEXT_SIZE,
            "batch_size": TABDPT_BATCH_SIZE,
        },
    },
    "tabpfn3_icl": {
        "environment": "tabpfn",
        "display_name": "TabPFN-3 Classifier",
        "condition": "in_context",
        "configuration": {
            "n_estimators": TABPFN_N_ESTIMATORS,
        },
    },
    "tabiclv2_icl": {
        "environment": "tabicl",
        "display_name": "TabICLv2 Classifier",
        "condition": "in_context",
        "configuration": {
            "n_estimators": TABICL_N_ESTIMATORS,
        },
    },
    "tabiclv2_finetuned": {
        "environment": "tabicl",
        "display_name": "TabICLv2 Classifier",
        "condition": "fine_tuned",
        "configuration": {
            "n_estimators": TABICL_N_ESTIMATORS,
            "fine_tune_epochs": TABICL_FINE_TUNE_EPOCHS,
            "fine_tune_time_limit": TABICL_FINE_TUNE_TIME_LIMIT,
            "fine_tune_patience": TABICL_FINE_TUNE_PATIENCE,
        },
    },
}

selected_foundation_conditions = []
if RUN_MITRA_ICL:
    selected_foundation_conditions.append("mitra_icl")
if RUN_TABDPT_ICL:
    selected_foundation_conditions.append("tabdpt_icl")
if RUN_TABPFN3_ICL:
    selected_foundation_conditions.append("tabpfn3_icl")
if RUN_TABICLV2_ICL:
    selected_foundation_conditions.append("tabiclv2_icl")
if RUN_MITRA_FINETUNED:
    selected_foundation_conditions.append("mitra_finetuned")
if RUN_TABICLV2_FINETUNED:
    selected_foundation_conditions.append("tabiclv2_finetuned")

run_plan = []
for key in selected_foundation_conditions:
    spec = CONDITION_SPECS[key]
    repo_spec = MODEL_REPOSITORIES[spec["environment"]]
    run_plan.append(
        {
            "model_key": key,
            "model": spec["display_name"],
            "condition": spec["condition"],
            "repository": repo_spec["repository"],
            "repository_commit": repo_spec["commit"],
            "weights_licence": repo_spec["licence"],
            "configuration": json.dumps(spec["configuration"], sort_keys=True),
        }
    )

display(pd.DataFrame(run_plan))


## 5.2 Build isolated Python 3.12 environments

This step:

1. clones each selected model repository at the pinned commit;
2. installs `uv` if it is not already available;
3. creates one Python 3.12 virtual environment per model family; and
4. installs that repository in editable mode using its own dependency contract.

A failure in one environment is reported without hiding successful environments. If **none** of the requested environments can be prepared, execution stops.


In [ ]:
# @title 5.2 Prepare or reuse model environments
FOUNDATION_PYTHON_SPEC = "3.12"
INSTALL_LOG_ROOT = WORKSPACE_ROOT / "install_logs"
INSTALL_LOG_ROOT.mkdir(exist_ok=True)

def shell_join(command):
    import shlex
    return " ".join(shlex.quote(str(part)) for part in command)

def stream_command(command, *, cwd=None, log_path=None):
    print("$", shell_join(command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1", "PIP_DISABLE_PIP_VERSION_CHECK": "1"},
    )
    output = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output.append(line)
    return_code = process.wait()
    if log_path is not None:
        Path(log_path).write_text("".join(output), encoding="utf-8")
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return "".join(output)

def ensure_uv() -> Path:
    found = shutil.which("uv")
    if found:
        return Path(found)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "uv"],
        check=True,
    )
    found = shutil.which("uv")
    if not found:
        candidate = Path(sys.executable).resolve().parent / "uv"
        if candidate.exists():
            found = str(candidate)
    if not found:
        raise RuntimeError("uv installed but its executable could not be located.")
    return Path(found)

def repository_path(environment_key: str) -> Path:
    spec = MODEL_REPOSITORIES[environment_key]
    return REPOSITORY_ROOT / f"{environment_key}-{spec['commit'][:12]}"

def environment_path(environment_key: str) -> Path:
    spec = MODEL_REPOSITORIES[environment_key]
    return ENVIRONMENT_ROOT / f"{environment_key}-{spec['commit'][:12]}"

def environment_python(path: Path) -> Path:
    return path / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

def checked_out_commit(path: Path):
    if not (path / ".git").exists():
        return None
    completed = subprocess.run(
        ["git", "-C", str(path), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
    )
    return completed.stdout.strip() if completed.returncode == 0 else None

def ensure_repository(environment_key: str) -> Path:
    spec = MODEL_REPOSITORIES[environment_key]
    destination = repository_path(environment_key)
    if destination.exists():
        actual = checked_out_commit(destination)
        if actual != spec["commit"]:
            raise RuntimeError(
                f"{environment_key}: cache has commit {actual}, expected {spec['commit']}."
            )
        return destination
    repository_url = f"https://github.com/{spec['repository']}.git"
    stream_command(
        ["git", "clone", "--filter=blob:none", "--no-checkout", repository_url, str(destination)]
    )
    stream_command(
        ["git", "-C", str(destination), "fetch", "--depth", "1", "origin", spec["commit"]]
    )
    stream_command(
        ["git", "-C", str(destination), "checkout", "--detach", spec["commit"]]
    )
    if checked_out_commit(destination) != spec["commit"]:
        raise RuntimeError(f"{environment_key}: repository checkout verification failed.")
    return destination

def ensure_environment(environment_key: str) -> Path:
    repository = ensure_repository(environment_key)
    destination = environment_path(environment_key)
    python_path = environment_python(destination)
    marker = destination / ".workshop-environment.json"
    expected = {
        "repository": MODEL_REPOSITORIES[environment_key]["repository"],
        "repository_commit": MODEL_REPOSITORIES[environment_key]["commit"],
        "python_spec": FOUNDATION_PYTHON_SPEC,
    }
    reusable = False
    if not REBUILD_MODEL_ENVIRONMENTS and python_path.exists() and marker.exists():
        try:
            reusable = json.loads(marker.read_text()) == expected
        except Exception:
            reusable = False
    if reusable:
        print(f"Reusing {environment_key}: {destination}")
        return python_path

    if destination.exists():
        destination = ENVIRONMENT_ROOT / f"{environment_key}-{uuid.uuid4().hex[:8]}"
        marker = destination / ".workshop-environment.json"
    uv_path = ensure_uv()
    log = INSTALL_LOG_ROOT / f"{environment_key}.log"
    stream_command(
        [
            str(uv_path),
            "venv",
            "--seed",
            "--python",
            FOUNDATION_PYTHON_SPEC,
            str(destination),
        ],
        log_path=log,
    )
    python_path = environment_python(destination)
    stream_command(
        [str(python_path), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
        log_path=log,
    )
    stream_command(
        [str(python_path), "-m", "pip", "install", "--editable", str(repository)],
        log_path=log,
    )
    import_name = MODEL_REPOSITORIES[environment_key]["import_name"]
    stream_command(
        [str(python_path), "-c", f"import {import_name}; print('imported {import_name}')"],
        log_path=log,
    )
    marker.write_text(json.dumps(expected, indent=2), encoding="utf-8")
    return python_path

selected_environment_keys = sorted(
    {CONDITION_SPECS[key]["environment"] for key in selected_foundation_conditions}
)
MODEL_ENVIRONMENT_PYTHONS = {}
ENVIRONMENT_ERRORS = {}

for environment_key in selected_environment_keys:
    try:
        MODEL_ENVIRONMENT_PYTHONS[environment_key] = ensure_environment(environment_key)
    except Exception as exc:
        ENVIRONMENT_ERRORS[environment_key] = f"{type(exc).__name__}: {exc}"
        print(f"ERROR preparing {environment_key}: {ENVIRONMENT_ERRORS[environment_key]}")

if ENVIRONMENT_ERRORS:
    display(pd.Series(ENVIRONMENT_ERRORS, name="error").to_frame())

requested_foundation_conditions = list(selected_foundation_conditions)
selected_foundation_conditions = [
    key
    for key in requested_foundation_conditions
    if CONDITION_SPECS[key]["environment"] in MODEL_ENVIRONMENT_PYTHONS
]

if requested_foundation_conditions and not selected_foundation_conditions:
    raise RuntimeError(
        "None of the requested foundation-model environments could be prepared."
    )

display(
    pd.Series(
        {key: str(value) for key, value in MODEL_ENVIRONMENT_PYTHONS.items()},
        name="python",
    ).to_frame()
)


## 5.3 Local foundation-model runner

The next cell writes a small runner program into the current workshop session. It is executed with the selected model's isolated Python interpreter.

The runner:

- reads only the fixed split CSVs;
- aligns every probability matrix to `low, mid, high`;
- records the exact model repository/checkpoint identity exposed by the pipeline;
- saves a fitted artifact generated in this runtime;
- validates on `val.csv`; and
- can later reload the saved artifact for the frozen test evaluation.

Serialized artifacts are trusted only because this notebook creates them in the current runtime. Do not replace them with external pickle files.


In [ ]:
# @title 5.3 Write the isolated model runner
RUNNER_PATH = SUPPORT_ROOT / "local_foundation_classifier.py"
RUNNER_SOURCE = "from __future__ import annotations\nimport argparse\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport pickle\nimport platform\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nTARGET = \"target\"\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\ndef package_version(name):\n    try:\n        return importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError:\n        return \"not installed\"\n\ndef resolve_device(preference):\n    import torch\n    if preference == \"cpu\":\n        return \"cpu\"\n    if preference == \"cuda\":\n        if not torch.cuda.is_available():\n            raise RuntimeError(\"CUDA requested but unavailable.\")\n        return \"cuda\"\n    return \"cuda\" if torch.cuda.is_available() else \"cpu\"\n\ndef align_probabilities(raw, model_classes, classes):\n    array = np.asarray(raw, dtype=float)\n    source = [str(value) for value in model_classes]\n    target = [str(value) for value in classes]\n    if array.ndim != 2 or array.shape[1] != len(source):\n        raise ValueError(\"Probability array does not match model classes.\")\n    mapping = {label: i for i, label in enumerate(source)}\n    missing = [label for label in target if label not in mapping]\n    if missing:\n        raise ValueError(f\"Model probabilities omit classes: {missing}\")\n    aligned = array[:, [mapping[label] for label in target]]\n    if not np.isfinite(aligned).all():\n        raise ValueError(\"Non-finite probabilities.\")\n    if not np.allclose(aligned.sum(axis=1), 1.0, rtol=1e-5, atol=1e-6):\n        raise ValueError(\"Probability rows do not sum to 1.\")\n    return aligned\n\ndef metrics(y_true, pred, proba, classes):\n    from sklearn.metrics import (\n        accuracy_score,\n        balanced_accuracy_score,\n        f1_score,\n        log_loss,\n        matthews_corrcoef,\n        roc_auc_score,\n    )\n    from sklearn.preprocessing import label_binarize\n\n    y = np.asarray([str(v) for v in y_true])\n    pred = np.asarray([str(v) for v in pred])\n    p = np.asarray(proba, dtype=float)\n    labels = [str(v) for v in classes]\n\n    label_to_index = {label: index for index, label in enumerate(labels)}\n    try:\n        y_index = np.asarray([label_to_index[value] for value in y], dtype=int)\n        pred_index = np.asarray([label_to_index[value] for value in pred], dtype=int)\n    except KeyError as exc:\n        raise ValueError(\n            f\"Unknown class label in targets/predictions: {exc.args[0]!r}\"\n        ) from exc\n\n    numeric_labels = list(range(len(labels)))\n    result = {\n        \"accuracy\": float(accuracy_score(y_index, pred_index)),\n        \"balanced_accuracy\": float(balanced_accuracy_score(y_index, pred_index)),\n        \"f1_macro\": float(\n            f1_score(\n                y_index,\n                pred_index,\n                labels=numeric_labels,\n                average=\"macro\",\n                zero_division=0,\n            )\n        ),\n        \"mcc\": float(matthews_corrcoef(y_index, pred_index)),\n        \"log_loss\": float(log_loss(y_index, p, labels=numeric_labels)),\n    }\n    try:\n        if len(labels) == 2:\n            result[\"roc_auc_ovr_macro\"] = float(\n                roc_auc_score(y_index, p[:, 1])\n            )\n        else:\n            y_binary = label_binarize(\n                y_index,\n                classes=np.arange(len(labels)),\n            )\n            result[\"roc_auc_ovr_macro\"] = float(\n                roc_auc_score(y_binary, p, average=\"macro\")\n            )\n    except ValueError:\n        result[\"roc_auc_ovr_macro\"] = float(\"nan\")\n    return result\n\ndef write_predictions(path, reference, pred, proba, classes):\n    out = pd.DataFrame({\n        \"row_id\": np.arange(len(reference)),\n        \"target\": reference[TARGET].astype(str).to_numpy(),\n        \"prediction\": np.asarray(pred).astype(str),\n    })\n    for i, label in enumerate(classes):\n        out[f\"probability_{label}\"] = np.asarray(proba, dtype=float)[:, i]\n    out.to_csv(path, index=False)\n\ndef save_pickle(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"wb\") as handle:\n        pickle.dump(value, handle)\n\ndef load_pickle(path):\n    with path.open(\"rb\") as handle:\n        return pickle.load(handle)\n\ndef run_mitra_development(config, train, val, out, device):\n    from mitra_pipeline import (\n        MODEL_ID,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        MitraClassificationPipeline,\n    )\n    repo = Path(config[\"repository_path\"])\n    weights_dir = repo / \"weights\" / \"mitra-classifier\"\n    settings = config[\"model_configuration\"]\n    pipe = MitraClassificationPipeline.from_pretrained(\n        weights_dir=weights_dir,\n        allow_download=True,\n        device=device,\n    )\n    artifact = out / \"artifact\" / \"mitra_predictor\"\n    artifact.parent.mkdir(parents=True, exist_ok=True)\n    t0 = time.perf_counter()\n    pipe.fit(\n        train[config[\"features\"] + [TARGET]],\n        target_column=TARGET,\n        eval_metric=\"balanced_accuracy\",\n        path=artifact,\n        fine_tune=bool(settings.get(\"fine_tune\", False)),\n        time_limit=int(settings.get(\"time_limit_seconds\", 600)),\n        seed=int(config[\"seed\"]),\n        problem_type=\"multiclass\",\n        fine_tune_steps=(\n            int(settings.get(\"fine_tune_steps\", 50))\n            if settings.get(\"fine_tune\", False)\n            else None\n        ),\n    )\n    fit_seconds = time.perf_counter() - t0\n    proba_df = pipe.predict_proba(val[config[\"features\"]])\n    model_classes = [str(value) for value in pipe.class_labels]\n    proba = align_probabilities(proba_df.to_numpy(dtype=float), model_classes, config[\"class_labels\"])\n    pred = np.asarray(config[\"class_labels\"], dtype=object)[np.argmax(proba, axis=1)]\n    metadata = {\n        \"artifact_type\": \"mitra_predictor\",\n        \"features\": config[\"features\"],\n        \"model_classes\": model_classes,\n    }\n    (out / \"artifact\" / \"metadata.json\").write_text(json.dumps(metadata, indent=2))\n    return {\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n        \"model_licence\": MODEL_LICENSE,\n        \"effective_mode\": \"fine_tuned\" if settings.get(\"fine_tune\", False) else \"in_context\",\n        \"fit_runtime_seconds\": fit_seconds,\n        \"classes\": model_classes,\n    }, pred, proba\n\ndef run_tabdpt_development(config, train, val, out, device):\n    from tabdpt_classifier_pipeline import export_artifact_bundle\n    from tabdpt_classifier_pipeline.pipeline import (\n        MODEL_ID,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        TabDPTClassificationPipeline,\n    )\n    repo = Path(config[\"repository_path\"])\n    weights_dir = repo / \"weights\" / \"tabdpt-1.2\"\n    settings = config[\"model_configuration\"]\n    pipe = TabDPTClassificationPipeline.from_pretrained(\n        weights_dir=weights_dir,\n        allow_download=True,\n        device=device,\n        use_flash=False,\n        compile_model=False,\n        seed=int(config[\"seed\"]),\n    )\n    t0 = time.perf_counter()\n    pipe.fit(\n        train[config[\"features\"] + [TARGET]],\n        target_column=TARGET,\n        seed=int(config[\"seed\"]),\n    )\n    fit_seconds = time.perf_counter() - t0\n    kwargs = {\n        \"n_ensembles\": int(settings.get(\"n_ensembles\", 4)),\n        \"context_size\": int(settings.get(\"context_size\", len(train))),\n        \"batch_size\": int(settings.get(\"batch_size\", 4096)),\n        \"seed\": int(config[\"seed\"]),\n    }\n    proba_df = pipe.predict_proba(val[config[\"features\"]], **kwargs)\n    model_classes = [str(value) for value in pipe.class_labels_]\n    proba = align_probabilities(proba_df.to_numpy(dtype=float), model_classes, config[\"class_labels\"])\n    pred = np.asarray(config[\"class_labels\"], dtype=object)[np.argmax(proba, axis=1)]\n    artifact_dir = out / \"artifact\" / \"tabdpt\"\n    manifest_path = export_artifact_bundle(\n        pipe,\n        train[config[\"features\"] + [TARGET]],\n        artifact_dir,\n    )\n    return {\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n        \"model_licence\": MODEL_LICENSE,\n        \"effective_mode\": \"in_context\",\n        \"fit_runtime_seconds\": fit_seconds,\n        \"classes\": model_classes,\n        \"context_size\": kwargs[\"context_size\"],\n        \"artifact_manifest\": manifest_path.name,\n    }, pred, proba\n\ndef run_tabpfn_development(config, train, val, out, device):\n    from tabpfn_classifier_pipeline.pipeline import (\n        MODEL_ID,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        TabPFNClassifierPipeline,\n    )\n    repo = Path(config[\"repository_path\"])\n    weights_dir = repo / \"weights\" / \"tabpfn-3-classifier\"\n    settings = config[\"model_configuration\"]\n    pipe = TabPFNClassifierPipeline.from_pretrained(\n        device=device,\n        weights_dir=weights_dir,\n        allow_download=True,\n        n_estimators=int(settings.get(\"n_estimators\", 4)),\n        random_state=int(config[\"seed\"]),\n    )\n    t0 = time.perf_counter()\n    pipe.fit(\n        train[config[\"features\"]],\n        train[TARGET],\n        target_column=TARGET,\n    )\n    fit_seconds = time.perf_counter() - t0\n    raw = pipe.predict_proba(val[config[\"features\"]])\n    model_classes = [str(value) for value in pipe.classes]\n    proba = align_probabilities(raw, model_classes, config[\"class_labels\"])\n    pred = np.asarray(config[\"class_labels\"], dtype=object)[np.argmax(proba, axis=1)]\n    artifact = out / \"artifact\" / \"tabpfn\"\n    pipe.save_artifact(artifact)\n    return {\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n        \"model_licence\": MODEL_LICENSE,\n        \"effective_mode\": \"in_context\",\n        \"fit_runtime_seconds\": fit_seconds,\n        \"classes\": model_classes,\n    }, pred, proba\n\ndef run_tabicl_development(config, train, val, out, device):\n    from tabicl_classifier_pipeline.api import (\n        MODEL_ID,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        TabICLClassificationPipeline,\n        apply_categorical_encoder,\n        create_classifier,\n        create_finetuned_classifier,\n        fine_tune_classifier,\n        fit_categorical_encoder,\n        prepare_classification_table,\n    )\n    repo = Path(config[\"repository_path\"])\n    weights_dir = repo / \"weights\" / \"tabicl-classifier-v2\"\n    settings = config[\"model_configuration\"]\n    train_clean, _ = prepare_classification_table(train[config[\"features\"] + [TARGET]], TARGET)\n    val_clean, _ = prepare_classification_table(\n        val[config[\"features\"] + [TARGET]],\n        TARGET,\n        min_rows=2,\n        min_classes=1,\n    )\n    encoders = fit_categorical_encoder(train_clean, config[\"features\"])\n    X_train, _ = apply_categorical_encoder(train_clean[config[\"features\"]], encoders)\n    X_val, _ = apply_categorical_encoder(val_clean[config[\"features\"]], encoders)\n\n    base = TabICLClassificationPipeline.from_pretrained(\n        weights_dir=weights_dir,\n        allow_download=True,\n        n_estimators=int(settings.get(\"n_estimators\", 8)),\n        random_state=int(config[\"seed\"]),\n        device=device,\n    )\n\n    t0 = time.perf_counter()\n    if config[\"condition\"] == \"fine_tuned\":\n        if device != \"cuda\":\n            raise RuntimeError(\"TabICLv2 fine-tuning requires CUDA.\")\n        ft_dir = out / \"model_work\" / \"finetune\"\n        ft_dir.mkdir(parents=True, exist_ok=True)\n        finetuner = create_finetuned_classifier(\n            epochs=int(settings.get(\"fine_tune_epochs\", 10)),\n            learning_rate=1e-5,\n            weight_decay=0.01,\n            n_estimators_finetune=1,\n            n_estimators_validation=1,\n            n_estimators_inference=int(settings.get(\"n_estimators\", 8)),\n            early_stopping=True,\n            patience=int(settings.get(\"fine_tune_patience\", 3)),\n            time_limit=int(settings.get(\"fine_tune_time_limit\", 600)),\n            eval_metric=\"accuracy\",\n            model_path=str(base.model_path),\n            allow_auto_download=False,\n            device=\"cuda\",\n            random_state=int(config[\"seed\"]),\n            verbose=True,\n            support_many_classes=True,\n        )\n        fine_tune_classifier(\n            finetuner,\n            X_train,\n            train_clean[TARGET],\n            X_val=X_val,\n            y_val=val_clean[TARGET],\n            output_dir=str(ft_dir),\n        )\n        best = ft_dir / \"best.ckpt\"\n        if not best.exists():\n            raise RuntimeError(\"TabICLv2 fine-tuning did not produce best.ckpt.\")\n        pipe = TabICLClassificationPipeline(\n            create_classifier(\n                model_path=best,\n                allow_auto_download=False,\n                n_estimators=int(settings.get(\"n_estimators\", 8)),\n                random_state=int(config[\"seed\"]),\n                device=device,\n            ),\n            model_path=best,\n            n_estimators=int(settings.get(\"n_estimators\", 8)),\n            random_state=int(config[\"seed\"]),\n            device=device,\n            source=\"fine-tuned\",\n        )\n    else:\n        pipe = base\n    pipe.fit(X_train, train_clean[TARGET])\n    fit_seconds = time.perf_counter() - t0\n    raw = pipe.predict_proba(X_val)\n    model_classes = [str(value) for value in pipe.classes_]\n    proba = align_probabilities(raw, model_classes, config[\"class_labels\"])\n    pred = np.asarray(config[\"class_labels\"], dtype=object)[np.argmax(proba, axis=1)]\n    save_pickle(out / \"artifact\" / \"tabicl.pkl\", {\"pipe\": pipe, \"encoders\": encoders})\n    return {\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n        \"model_licence\": MODEL_LICENSE,\n        \"effective_mode\": \"fine_tuned\" if config[\"condition\"] == \"fine_tuned\" else \"in_context\",\n        \"fit_runtime_seconds\": fit_seconds,\n        \"classes\": model_classes,\n    }, pred, proba\n\ndef run_test(config, test, out, device):\n    source = Path(config[\"source_run_dir\"])\n    family = config[\"model_family\"]\n    features = config[\"features\"]\n    classes = config[\"class_labels\"]\n\n    if family == \"mitra\":\n        from autogluon.tabular import TabularPredictor\n        predictor = TabularPredictor.load(str(source / \"artifact\" / \"mitra_predictor\"))\n        raw = predictor.predict_proba(test[features])\n        model_classes = [str(value) for value in predictor.class_labels]\n        raw_values = raw.to_numpy(dtype=float)\n    elif family == \"tabdpt\":\n        from tabdpt_classifier_pipeline import load_verified_artifact\n        repo = Path(config[\"repository_path\"])\n        settings = config[\"model_configuration\"]\n        pipe = load_verified_artifact(\n            source / \"artifact\" / \"tabdpt\" / \"artifact.json\",\n            model_weight_path=repo / \"weights\" / \"tabdpt-1.2\" / \"tabdpt1_2.safetensors\",\n            device=device,\n            use_flash=False,\n            compile_model=False,\n            seed=int(config[\"seed\"]),\n        )\n        kwargs = {\n            \"n_ensembles\": int(settings.get(\"n_ensembles\", 4)),\n            \"context_size\": int(settings.get(\"context_size\", 4180)),\n            \"batch_size\": int(settings.get(\"batch_size\", 4096)),\n            \"seed\": int(config[\"seed\"]),\n        }\n        raw = pipe.predict_proba(test[features], **kwargs)\n        model_classes = [str(value) for value in pipe.class_labels_]\n        raw_values = raw.to_numpy(dtype=float)\n    elif family == \"tabpfn\":\n        from tabpfn_classifier_pipeline.pipeline import TabPFNClassifierPipeline\n        pipe = TabPFNClassifierPipeline.from_artifact(\n            source / \"artifact\" / \"tabpfn\",\n            device=device,\n        )\n        raw_values = pipe.predict_proba(test[features])\n        model_classes = [str(value) for value in pipe.classes]\n    elif family == \"tabicl\":\n        from tabicl_classifier_pipeline.api import apply_categorical_encoder\n        restored = load_pickle(source / \"artifact\" / \"tabicl.pkl\")\n        pipe, encoders = restored[\"pipe\"], restored[\"encoders\"]\n        X_test, _ = apply_categorical_encoder(test[features], encoders)\n        raw_values = pipe.predict_proba(X_test)\n        model_classes = [str(value) for value in pipe.classes_]\n    else:\n        raise ValueError(f\"Unknown model family: {family}\")\n\n    proba = align_probabilities(raw_values, model_classes, classes)\n    pred = np.asarray(classes, dtype=object)[np.argmax(proba, axis=1)]\n    return pred, proba\n\ndef execute(config):\n    out = Path(config[\"output_dir\"])\n    out.mkdir(parents=True, exist_ok=True)\n    seed_all(int(config[\"seed\"]))\n    device = resolve_device(config[\"device_preference\"])\n    split_paths = {key: Path(value) for key, value in config[\"split_paths\"].items()}\n\n    if config[\"phase\"] == \"development\":\n        train = pd.read_csv(split_paths[\"train\"], float_precision=\"round_trip\")\n        val = pd.read_csv(split_paths[\"val\"], float_precision=\"round_trip\")\n        family = config[\"model_family\"]\n        if family == \"mitra\":\n            identity, pred, proba = run_mitra_development(config, train, val, out, device)\n        elif family == \"tabdpt\":\n            identity, pred, proba = run_tabdpt_development(config, train, val, out, device)\n        elif family == \"tabpfn\":\n            identity, pred, proba = run_tabpfn_development(config, train, val, out, device)\n        elif family == \"tabicl\":\n            identity, pred, proba = run_tabicl_development(config, train, val, out, device)\n        else:\n            raise ValueError(f\"Unknown model family: {family}\")\n        common_metrics = metrics(val[TARGET], pred, proba, config[\"class_labels\"])\n        write_predictions(\n            out / \"validation_predictions.csv\",\n            val,\n            pred,\n            proba,\n            config[\"class_labels\"],\n        )\n        result = {\n            **identity,\n            \"model_key\": config[\"model_key\"],\n            \"model\": config[\"display_name\"],\n            \"family\": \"Tabular foundation model\",\n            \"condition\": config[\"condition\"],\n            \"partition\": \"validation\",\n            \"device\": device,\n            \"repository\": config[\"repository\"],\n            \"repository_commit\": config[\"repository_commit\"],\n            \"feature_columns\": config[\"features\"],\n            \"metrics\": common_metrics,\n            \"runtime\": {\n                \"python\": platform.python_version(),\n                \"torch\": package_version(\"torch\"),\n                \"pandas\": package_version(\"pandas\"),\n                \"numpy\": package_version(\"numpy\"),\n                \"scikit_learn\": package_version(\"scikit-learn\"),\n            },\n        }\n        (out / \"run_config.json\").write_text(json.dumps(config, indent=2))\n        (out / \"result.json\").write_text(json.dumps(result, indent=2, default=str))\n        print(json.dumps({\"model\": result[\"model\"], \"condition\": result[\"condition\"], \"metrics\": common_metrics}, indent=2))\n    elif config[\"phase\"] == \"test\":\n        test = pd.read_csv(split_paths[\"test\"], float_precision=\"round_trip\")\n        t0 = time.perf_counter()\n        pred, proba = run_test(config, test, out, device)\n        prediction_seconds = time.perf_counter() - t0\n        common_metrics = metrics(test[TARGET], pred, proba, config[\"class_labels\"])\n        write_predictions(\n            out / \"test_predictions.csv\",\n            test,\n            pred,\n            proba,\n            config[\"class_labels\"],\n        )\n        result = {\n            \"model_key\": config[\"model_key\"],\n            \"model\": config[\"display_name\"],\n            \"family\": \"Tabular foundation model\",\n            \"condition\": config[\"condition\"],\n            \"partition\": \"test\",\n            \"device\": device,\n            \"repository\": config[\"repository\"],\n            \"repository_commit\": config[\"repository_commit\"],\n            \"feature_columns\": config[\"features\"],\n            \"metrics\": common_metrics,\n            \"evaluation_protocol\": \"reload_saved_development_artifact_no_gradient_refit\",\n            \"prediction_runtime_seconds\": prediction_seconds,\n        }\n        (out / \"run_config.json\").write_text(json.dumps(config, indent=2))\n        (out / \"result.json\").write_text(json.dumps(result, indent=2, default=str))\n        print(json.dumps({\"model\": result[\"model\"], \"metrics\": common_metrics}, indent=2))\n    else:\n        raise ValueError(\"phase must be development or test\")\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", required=True)\n    args = parser.parse_args()\n    execute(json.loads(Path(args.config).read_text()))\n\nif __name__ == \"__main__\":\n    main()\n"
RUNNER_PATH.write_text(RUNNER_SOURCE, encoding="utf-8")
compile(RUNNER_SOURCE, str(RUNNER_PATH), "exec")
RUNNER_SHA256 = sha256_file(RUNNER_PATH)
print("Runner ready:", RUNNER_PATH)
print("Runner SHA-256:", RUNNER_SHA256)


In [ ]:
# @title 5.4 Execute validation-stage foundation-model runs
FOUNDATION_ROOT = RUN_ROOT / "foundation"
FOUNDATION_ROOT.mkdir(exist_ok=True)

def run_foundation_condition(
    key: str,
    *,
    features=None,
    output_key=None,
    condition_override=None,
):
    spec = CONDITION_SPECS[key]
    env_key = spec["environment"]
    if env_key not in MODEL_ENVIRONMENT_PYTHONS:
        raise RuntimeError(f"Environment {env_key!r} is unavailable.")

    repo_spec = MODEL_REPOSITORIES[env_key]
    run_key = output_key or key
    selected_features = list(features or FEATURE_COLUMNS)

    run_identity = {
        "model_key": run_key,
        "base_condition_key": key,
        "model_family": env_key,
        "display_name": spec["display_name"],
        "condition": condition_override or spec["condition"],
        "model_configuration": spec["configuration"],
        "repository": repo_spec["repository"],
        "repository_commit": repo_spec["commit"],
        "features": selected_features,
        "class_labels": CLASS_LABELS,
        "seed": RANDOM_SEED,
        "device_preference": DEVICE_PREFERENCE,
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "runner_sha256": RUNNER_SHA256,
    }
    run_fingerprint = fingerprint(run_identity)
    output_dir = FOUNDATION_ROOT / f"{run_key}-{run_fingerprint[:12]}"

    config = {
        "phase": "development",
        **run_identity,
        "repository_path": str(repository_path(env_key)),
        "split_paths": {name: str(value) for name, value in staged_paths.items()},
        "output_dir": str(output_dir),
        "run_fingerprint": run_fingerprint,
    }

    if output_dir.exists():
        result_path = output_dir / "result.json"
        prediction_path = output_dir / "validation_predictions.csv"
        config_path_existing = output_dir / "run_config.json"
        if (
            not FORCE_MODEL_RERUN
            and result_path.exists()
            and prediction_path.exists()
            and config_path_existing.exists()
            and json.loads(config_path_existing.read_text()) == config
        ):
            print(f"Reusing identity-matched run: {output_dir.name}")
            return output_dir
        if FORCE_MODEL_RERUN:
            shutil.rmtree(output_dir)
        elif result_path.exists() or prediction_path.exists() or config_path_existing.exists():
            raise RuntimeError(
                f"Run cache at {output_dir} is incomplete or does not match the requested identity."
            )

    config_path = SUPPORT_ROOT / f"{run_key}-{run_fingerprint[:12]}.json"
    write_json(config_path, config)
    python_path = MODEL_ENVIRONMENT_PYTHONS[env_key]
    log_path = FOUNDATION_ROOT / f"{run_key}-{run_fingerprint[:12]}.log"
    stream_command(
        [str(python_path), str(RUNNER_PATH), "--config", str(config_path)],
        log_path=log_path,
    )

    persisted = json.loads((output_dir / "run_config.json").read_text())
    if persisted != config:
        raise RuntimeError("Foundation run persisted a configuration that differs from the requested identity.")
    return output_dir

FOUNDATION_RUN_DIRS = {}
FOUNDATION_RUN_ERRORS = {}

for key in selected_foundation_conditions:
    print(f"\n=== {key} ===")
    try:
        FOUNDATION_RUN_DIRS[key] = run_foundation_condition(key)
    except Exception as exc:
        FOUNDATION_RUN_ERRORS[key] = f"{type(exc).__name__}: {exc}"
        print("ERROR:", FOUNDATION_RUN_ERRORS[key])

if FOUNDATION_RUN_ERRORS:
    display(pd.Series(FOUNDATION_RUN_ERRORS, name="error").to_frame())

if selected_foundation_conditions and not FOUNDATION_RUN_DIRS:
    raise RuntimeError(
        "All selected foundation-model validation runs failed. "
        "The notebook will not present a baseline-only result as the multi-model exercise."
    )

if len(selected_foundation_conditions) >= 2 and len(FOUNDATION_RUN_DIRS) < 2:
    print(
        "WARNING: fewer than two foundation-model conditions succeeded; "
        "the participant checklist is not yet satisfied."
    )

print(
    f"Successful foundation-model runs: "
    f"{len(FOUNDATION_RUN_DIRS)} / {len(selected_foundation_conditions)}"
)


In [ ]:
# @title 5.5 Load, validate, and compare foundation-model results
foundation_records = []
foundation_prediction_registry = {}

for key, run_dir in FOUNDATION_RUN_DIRS.items():
    result = json.loads((run_dir / "result.json").read_text())
    predictions = pd.read_csv(run_dir / "validation_predictions.csv")
    if len(predictions) != len(frames["val"]):
        raise ValueError(f"{key}: validation prediction row count mismatch.")
    if predictions["row_id"].tolist() != list(range(len(frames["val"]))):
        raise ValueError(f"{key}: invalid row_id sequence.")
    if not np.array_equal(
        predictions["target"].astype(str).to_numpy(),
        y_val.to_numpy(),
    ):
        raise ValueError(f"{key}: prediction targets do not match validation targets.")

    record = {
        "model_key": key,
        "model": result["model"],
        "family": result["family"],
        "condition": result["condition"],
        **result["metrics"],
        "runtime_seconds": result.get("fit_runtime_seconds"),
        "device": result["device"],
        "licence": MODEL_REPOSITORIES[CONDITION_SPECS[key]["environment"]]["licence"],
        "notes": "",
    }
    foundation_records.append(record)
    foundation_prediction_registry[key] = predictions

foundation_validation_results = pd.DataFrame(foundation_records)

combined_validation_results = pd.concat(
    [baseline_validation_results, foundation_validation_results],
    ignore_index=True,
    sort=False,
).sort_values(PRIMARY_METRIC, ascending=False).reset_index(drop=True)

display(
    combined_validation_results[
        [
            "model",
            "condition",
            "balanced_accuracy",
            "accuracy",
            "f1_macro",
            "mcc",
            "log_loss",
            "roc_auc_ovr_macro",
            "device",
        ]
    ].round(4)
)

fig, ax = plt.subplots(figsize=(10, 6))
plot_data = combined_validation_results.sort_values(PRIMARY_METRIC)
ax.barh(
    plot_data["model"].astype(str) + " / " + plot_data["condition"].astype(str),
    plot_data[PRIMARY_METRIC],
)
ax.set_title("Validation balanced accuracy")
ax.set_xlabel("Balanced accuracy")
plt.tight_layout()
comparison_figure = FIGURE_ROOT / "validation_balanced_accuracy.png"
plt.savefig(comparison_figure, dpi=140)
plt.show()


## Multi-model checkpoint

Use the validation comparison to answer:

1. Which conditions improve on the training-prior majority baseline?
2. Does the ordering change between ordinary accuracy and balanced accuracy?
3. Does the model with the highest balanced accuracy also have strong macro-F1?
4. Is a model achieving higher correctness by becoming overconfident, as reflected in log loss?
5. Which demand band has the lowest recall?
6. How much runtime and licensing complexity accompanies any improvement?

Avoid writing “Model X is the best classifier.” The evidence supports only a statement about this dataset artifact, split, metric suite, repository revision, and run configuration.


# 6. Inspect confusion and class-specific recall

A single headline metric can hide systematic failure on one demand band. The next cell uses validation predictions only.


In [ ]:
# @title 6.1 Confusion matrix for one successful model
MODEL_TO_INSPECT = "" # @param {type:"string"}

validation_registry = {
    **baseline_prediction_registry,
    **foundation_prediction_registry,
}
if not validation_registry:
    raise RuntimeError("No validation predictions are available.")

if MODEL_TO_INSPECT and MODEL_TO_INSPECT not in validation_registry:
    raise ValueError(
        f"{MODEL_TO_INSPECT!r} has no validation predictions. "
        f"Choose one of {sorted(validation_registry)}."
    )

if not MODEL_TO_INSPECT:
    ranked_keys = combined_validation_results["model_key"].tolist()
    MODEL_TO_INSPECT = next(key for key in ranked_keys if key in validation_registry)

diagnostic = validation_registry[MODEL_TO_INSPECT]
cm = confusion_matrix(
    diagnostic["target"].astype(str),
    diagnostic["prediction"].astype(str),
    labels=CLASS_LABELS,
)
cm_frame = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in CLASS_LABELS],
    columns=[f"pred_{label}" for label in CLASS_LABELS],
)
display(cm_frame)

recalls = recall_score(
    diagnostic["target"].astype(str),
    diagnostic["prediction"].astype(str),
    labels=CLASS_LABELS,
    average=None,
    zero_division=0,
)
display(
    pd.DataFrame(
        {"class": CLASS_LABELS, "recall": recalls}
    ).round(4)
)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(cm)
ax.set_xticks(range(len(CLASS_LABELS)), CLASS_LABELS)
ax.set_yticks(range(len(CLASS_LABELS)), CLASS_LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("Observed")
ax.set_title(f"Validation confusion matrix: {MODEL_TO_INSPECT}")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
confusion_figure = FIGURE_ROOT / f"confusion_{MODEL_TO_INSPECT}.png"
plt.savefig(confusion_figure, dpi=140)
plt.show()


# 7. Stockout-feature ablation

The stockout columns can carry predictive signal, but an ablation is still only predictive evidence. Removing them and observing a performance change does **not** establish that stockouts causally change demand.

The classical ablation is enabled by default. The foundation-model ablation is optional because it can require another expensive checkpoint run.


In [ ]:
# @title 7.1 Classical and optional foundation-model stockout ablation
STOCKOUT_FEATURES = ["stockout_hours", "roll_7_stockout"]
REDUCED_FEATURES = [c for c in FEATURE_COLUMNS if c not in STOCKOUT_FEATURES]

RUN_CLASSICAL_ABLATION = True # @param {type:"boolean"}
RUN_FOUNDATION_ABLATION = False # @param {type:"boolean"}
FOUNDATION_ABLATION_CONDITION = "" # @param {type:"string"}

ablation_records = []

if RUN_CLASSICAL_ABLATION:
    reduced_lgbm = LGBMClassifier(
        n_estimators=300,
        random_state=RANDOM_SEED,
        verbosity=-1,
    )
    reduced_lgbm.fit(frames["train"][REDUCED_FEATURES], y_train)
    reduced_proba = align_probabilities(
        reduced_lgbm.predict_proba(frames["val"][REDUCED_FEATURES]),
        reduced_lgbm.classes_,
        CLASS_LABELS,
    )
    reduced_pred = np.asarray(CLASS_LABELS, dtype=object)[np.argmax(reduced_proba, axis=1)]
    reduced_metrics = classification_metrics(y_val, reduced_pred, reduced_proba, CLASS_LABELS)
    full = baseline_validation_results.loc[
        baseline_validation_results["model_key"] == "lightgbm"
    ].iloc[0]
    ablation_records.append(
        {
            "model": "LightGBM",
            "comparison": "remove stockout features",
            "full_balanced_accuracy": full["balanced_accuracy"],
            "ablated_balanced_accuracy": reduced_metrics["balanced_accuracy"],
            "delta": reduced_metrics["balanced_accuracy"] - full["balanced_accuracy"],
        }
    )

FOUNDATION_ABLATION_RUN_DIR = None
if RUN_FOUNDATION_ABLATION:
    candidates = list(FOUNDATION_RUN_DIRS)
    if not candidates:
        print("No successful foundation model is available for ablation.")
    else:
        base_key = FOUNDATION_ABLATION_CONDITION or candidates[0]
        if base_key not in FOUNDATION_RUN_DIRS:
            raise ValueError(
                f"Choose a successful foundation condition: {sorted(FOUNDATION_RUN_DIRS)}"
            )
        ablation_key = f"ablation_{base_key}_no_stockout"
        try:
            FOUNDATION_ABLATION_RUN_DIR = run_foundation_condition(
                base_key,
                features=REDUCED_FEATURES,
                output_key=ablation_key,
                condition_override=CONDITION_SPECS[base_key]["condition"] + "_no_stockout",
            )
            result = json.loads(
                (FOUNDATION_ABLATION_RUN_DIR / "result.json").read_text()
            )
            full_result = json.loads(
                (FOUNDATION_RUN_DIRS[base_key] / "result.json").read_text()
            )
            ablation_records.append(
                {
                    "model": result["model"],
                    "comparison": "remove stockout features",
                    "full_balanced_accuracy": full_result["metrics"]["balanced_accuracy"],
                    "ablated_balanced_accuracy": result["metrics"]["balanced_accuracy"],
                    "delta": (
                        result["metrics"]["balanced_accuracy"]
                        - full_result["metrics"]["balanced_accuracy"]
                    ),
                }
            )
        except Exception as exc:
            print("Foundation ablation failed:", f"{type(exc).__name__}: {exc}")

if ablation_records:
    display(pd.DataFrame(ablation_records).round(4))
else:
    print("No ablation was executed.")


# 8. Freeze the experiment before evaluating test labels

Validation data is for development and selection. The test partition is for the final measurement after settings are fixed.

The freeze records:

- dataset and split hashes;
- selected successful model keys;
- validation metrics;
- artifact digests where applicable;
- feature columns;
- class order; and
- the primary metric.

Turn **Freeze now** on only after you are done changing model conditions.


In [ ]:
# @title 8.0 Freeze selected successful full-feature runs
FREEZE_NOW = False # @param {type:"boolean"}
SELECTED_MODEL_KEYS = "" # @param {type:"string"}
# @markdown Comma-separated keys. Blank selects all successful full-feature baseline and foundation runs.

full_feature_keys = (
    baseline_validation_results["model_key"].tolist()
    + list(FOUNDATION_RUN_DIRS)
)

selected_keys = (
    [value.strip() for value in SELECTED_MODEL_KEYS.split(",") if value.strip()]
    or full_feature_keys
)

unknown = [key for key in selected_keys if key not in full_feature_keys]
if unknown:
    raise ValueError(f"Unknown or unsuccessful model keys: {unknown}")

FREEZE_PATH = SESSION_ROOT / "freeze.json"

if FREEZE_NOW:
    selected_manifest = {}
    for key in selected_keys:
        if key in baseline_validation_results["model_key"].values:
            run_dir = BASELINE_ROOT / key
            result = json.loads((run_dir / "result.json").read_text())
            selected_manifest[key] = {
                "backend": "baseline",
                "run_dir": str(run_dir),
                "result_sha256": sha256_file(run_dir / "result.json"),
                "artifact_sha256": sha256_file(run_dir / "artifact.pkl"),
                "validation": {
                    metric: result.get(metric)
                    for metric in (
                        "accuracy",
                        "balanced_accuracy",
                        "f1_macro",
                        "mcc",
                        "log_loss",
                        "roc_auc_ovr_macro",
                    )
                },
            }
        else:
            run_dir = FOUNDATION_RUN_DIRS[key]
            result = json.loads((run_dir / "result.json").read_text())
            selected_manifest[key] = {
                "backend": "foundation",
                "run_dir": str(run_dir),
                "environment": CONDITION_SPECS[key]["environment"],
                "result_sha256": sha256_file(run_dir / "result.json"),
                "run_config_sha256": sha256_file(run_dir / "run_config.json"),
                "artifact_inventory": file_inventory(run_dir / "artifact"),
                "validation": result["metrics"],
            }

    freeze_document = {
        "workshop_revision": WORKSHOP_REVISION,
        "dataset_identity": DATASET_IDENTITY,
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "primary_metric": PRIMARY_METRIC,
        "class_labels": CLASS_LABELS,
        "feature_columns": FEATURE_COLUMNS,
        "selected": selected_manifest,
        "protocol": "validation-selection-then-reload-saved-artifacts-for-test",
    }
    write_json(FREEZE_PATH, freeze_document)
    print("Frozen:", FREEZE_PATH)
    print("Selected:", selected_keys)
else:
    print("Not frozen yet.")
    print("Available successful keys:", full_feature_keys)


In [ ]:
# @title 8.1 Evaluate only frozen models on the test partition
EVALUATE_FROZEN_TEST = False # @param {type:"boolean"}

def evaluate_baseline_test(key: str):
    run_dir = BASELINE_ROOT / key
    with (run_dir / "artifact.pkl").open("rb") as handle:
        artifact = pickle.load(handle)
    test = frames["test"]
    X = test[FEATURE_COLUMNS]
    y = test[TARGET_COLUMN].astype(str)
    kind = artifact["kind"]

    if kind == "majority":
        priors = pd.Series(artifact["priors"]).reindex(CLASS_LABELS).fillna(0.0)
        pred = np.repeat(artifact["majority_label"], len(test))
        proba = np.tile(priors.to_numpy(dtype=float), (len(test), 1))
    elif kind == "lag7_quantile":
        edges = np.asarray(artifact["edges"], dtype=float)
        ids = np.digitize(X[artifact["feature"]].to_numpy(dtype=float), edges)
        pred = np.asarray(CLASS_LABELS, dtype=object)[ids]
        proba = np.full((len(test), len(CLASS_LABELS)), 1e-6)
        proba[np.arange(len(test)), ids] = 1.0 - (len(CLASS_LABELS) - 1) * 1e-6
    elif kind == "sklearn":
        model = artifact["model"]
        raw = model.predict_proba(X)
        if isinstance(model, Pipeline):
            model_classes = model.named_steps["model"].classes_
        else:
            model_classes = model.classes_
        proba = align_probabilities(raw, model_classes, CLASS_LABELS)
        pred = np.asarray(CLASS_LABELS, dtype=object)[np.argmax(proba, axis=1)]
    else:
        raise ValueError(f"Unknown baseline artifact kind: {kind}")

    return classification_metrics(y, pred, proba, CLASS_LABELS), prediction_frame(
        test, pred, proba, CLASS_LABELS
    )

TEST_RESULTS = None
TEST_PREDICTION_REGISTRY = {}

if EVALUATE_FROZEN_TEST:
    if not FREEZE_PATH.exists():
        raise ValueError("Freeze the development experiment in Section 8.0 first.")
    frozen = json.loads(FREEZE_PATH.read_text())
    if frozen["dataset_fingerprint"] != DATASET_FINGERPRINT:
        raise ValueError("Dataset identity changed after freeze.")

    test_records = []
    for key, entry in frozen["selected"].items():
        print(f"\n=== test: {key} ===")
        run_dir = Path(entry["run_dir"])
        if sha256_file(run_dir / "result.json") != entry["result_sha256"]:
            raise ValueError(f"{key}: frozen development result changed after freeze.")

        if entry["backend"] == "baseline":
            if sha256_file(run_dir / "artifact.pkl") != entry["artifact_sha256"]:
                raise ValueError(f"{key}: frozen baseline artifact changed after freeze.")
            metrics_value, predictions = evaluate_baseline_test(key)
            model_row = combined_validation_results.loc[
                combined_validation_results["model_key"] == key
            ].iloc[0]
            test_records.append(
                {
                    "model_key": key,
                    "model": model_row["model"],
                    "condition": model_row["condition"],
                    **metrics_value,
                    "device": "cpu",
                }
            )
            TEST_PREDICTION_REGISTRY[key] = predictions
        else:
            if sha256_file(run_dir / "run_config.json") != entry["run_config_sha256"]:
                raise ValueError(f"{key}: frozen foundation run configuration changed after freeze.")
            verify_file_inventory(run_dir / "artifact", entry["artifact_inventory"])
            env_key = entry["environment"]
            spec = CONDITION_SPECS[key]
            repo_spec = MODEL_REPOSITORIES[env_key]
            output_dir = RUN_ROOT / "test" / key
            output_dir.mkdir(parents=True, exist_ok=True)
            config = {
                "phase": "test",
                "model_key": key,
                "base_condition_key": key,
                "model_family": env_key,
                "display_name": spec["display_name"],
                "condition": spec["condition"],
                "model_configuration": spec["configuration"],
                "repository": repo_spec["repository"],
                "repository_commit": repo_spec["commit"],
                "repository_path": str(repository_path(env_key)),
                "features": FEATURE_COLUMNS,
                "class_labels": CLASS_LABELS,
                "seed": RANDOM_SEED,
                "device_preference": DEVICE_PREFERENCE,
                "dataset_fingerprint": DATASET_FINGERPRINT,
                "runner_sha256": RUNNER_SHA256,
                "split_paths": {name: str(path) for name, path in staged_paths.items()},
                "source_run_dir": entry["run_dir"],
                "output_dir": str(output_dir),
            }
            config_path = SUPPORT_ROOT / f"test-{key}.json"
            write_json(config_path, config)
            stream_command(
                [
                    str(MODEL_ENVIRONMENT_PYTHONS[env_key]),
                    str(RUNNER_PATH),
                    "--config",
                    str(config_path),
                ],
                log_path=RUN_ROOT / "test" / f"{key}.log",
            )
            result = json.loads((output_dir / "result.json").read_text())
            predictions = pd.read_csv(output_dir / "test_predictions.csv")
            test_records.append(
                {
                    "model_key": key,
                    "model": result["model"],
                    "condition": result["condition"],
                    **result["metrics"],
                    "device": result["device"],
                }
            )
            TEST_PREDICTION_REGISTRY[key] = predictions

    TEST_RESULTS = pd.DataFrame(test_records).sort_values(
        PRIMARY_METRIC, ascending=False
    ).reset_index(drop=True)
    display(
        TEST_RESULTS[
            [
                "model",
                "condition",
                "balanced_accuracy",
                "accuracy",
                "f1_macro",
                "mcc",
                "log_loss",
                "roc_auc_ovr_macro",
                "device",
            ]
        ].round(4)
    )
else:
    print("Test labels remain unevaluated. Enable EVALUATE_FROZEN_TEST only after freezing.")


## Write an evidence-based conclusion

Use this structure after the frozen test evaluation:

### 1. State the question

> We tested whether selected tabular foundation-model classifiers improved seven-day-ahead retail demand-band classification relative to heuristic and classical baselines.

### 2. Report the primary result

> On the fixed test partition, **[model and condition]** obtained balanced accuracy **[value]**, compared with **[value]** for **[reference baseline]**.

### 3. Report supporting evidence

Include accuracy, macro-F1, MCC and log loss. If ROC-AUC is reported, identify it as one-vs-rest macro AUC.

### 4. Report class-specific behavior

State which of `low`, `mid`, or `high` had the weakest recall or was most frequently confused.

### 5. Describe adaptation honestly

Say whether the selected foundation model used **in-context conditioning** or **gradient fine-tuning**.

### 6. State limitations

At minimum:

- the sample covers only a subset of FreshRetailNet-50K;
- classes are training-derived demand bands, not intrinsic business categories;
- the target is observed sales and can be censored by stockouts;
- one chronological split does not quantify variance across time periods;
- probabilities are not assumed calibrated;
- TabPFN-3 carries non-commercial weight restrictions; and
- results do not establish production readiness, causal effects, or benchmark superiority.


## Participant submission checklist

- [ ] Dataset repository revision and SHA-256
- [ ] Train/validation row counts and class proportions
- [ ] One development-only EDA visualization
- [ ] Classical baseline table
- [ ] At least two successful foundation-model conditions
- [ ] Validation comparison using balanced accuracy
- [ ] Confusion matrix and per-class recall
- [ ] Stockout ablation result or a justified reason for omitting it
- [ ] Freeze manifest
- [ ] Frozen test comparison table
- [ ] Effective adaptation mode for each selected foundation model
- [ ] Dataset and model licence notes
- [ ] One evidence-based conclusion
- [ ] At least three limitations


# 9. Export the current experiment record

The export contains results and metadata, not the foundation-model weight files themselves.


In [ ]:
# @title 9.1 Export results, figures, provenance, and freeze metadata
CREATE_ZIP_BUNDLE = True # @param {type:"boolean"}

export_dir = EXPORT_ROOT / f"report-{uuid.uuid4().hex[:8]}"
export_dir.mkdir(parents=True, exist_ok=False)

baseline_validation_results.to_csv(
    export_dir / "baseline_validation_results.csv",
    index=False,
)
foundation_validation_results.to_csv(
    export_dir / "foundation_validation_results.csv",
    index=False,
)
combined_validation_results.to_csv(
    export_dir / "combined_validation_results.csv",
    index=False,
)

if TEST_RESULTS is not None:
    TEST_RESULTS.to_csv(export_dir / "frozen_test_results.csv", index=False)

if FREEZE_PATH.exists():
    shutil.copy2(FREEZE_PATH, export_dir / "freeze.json")

figure_dir = export_dir / "figures"
figure_dir.mkdir()
for path in FIGURE_ROOT.glob("*.png"):
    shutil.copy2(path, figure_dir / path.name)

manifest = {
    "workshop_revision": WORKSHOP_REVISION,
    "dataset_provenance": DATASET_PROVENANCE,
    "dataset_identity": DATASET_IDENTITY,
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "software_versions": SOFTWARE_VERSIONS,
    "primary_metric": PRIMARY_METRIC,
    "class_labels": CLASS_LABELS,
    "feature_columns": FEATURE_COLUMNS,
    "successful_foundation_conditions": list(FOUNDATION_RUN_DIRS),
    "foundation_failures": FOUNDATION_RUN_ERRORS,
    "includes_frozen_test_results": TEST_RESULTS is not None,
    "model_repositories": MODEL_REPOSITORIES,
    "artifact_policy": (
        "Foundation-model artifacts remain in the notebook runtime; "
        "the report contains provenance and metrics, not redistributed model weights."
    ),
}
write_json(export_dir / "experiment_manifest.json", manifest)

print("Export directory:", export_dir)
if CREATE_ZIP_BUNDLE:
    bundle = Path(
        shutil.make_archive(
            str(export_dir),
            "zip",
            root_dir=export_dir,
        )
    )
    print("Report ZIP:", bundle)
    print("SHA-256:", sha256_file(bundle))


# Appendix. References and provenance

## Dataset

Workshop artifact: `kurtvalcorza/mitra-classifier-pipeline/examples/sample-data/freshretailnet-band-h7.zip`  
Pinned repository revision: `469d91252f3583b38d08b5c4d90fef4848b93f24`  
Pinned archive SHA-256: `ad2d2a8729749bb055754e4867acfb048fc27816f4eb344961d962b59c0be6dd`  
Source dataset: `Dingdong-Inc/FreshRetailNet-50K`  
Dataset licence: CC BY 4.0

The target bands are generated by `examples/build_freshretailnet_dataset.py`: the future seven-day-ahead `sale_amount` is computed first, the temporal train/validation/test split is formed with seven-row embargoes, then the band edges are learned from the training rows only and applied to all partitions.

## Pinned classifier pipeline repositories

| Model | Repository | Commit | Checkpoint |
|---|---|---|---|
| Mitra Classifier | `kurtvalcorza/mitra-classifier-pipeline` | `469d91252f3583b38d08b5c4d90fef4848b93f24` | `autogluon/mitra-classifier` @ `c425e9fa0910a6be1c494321792e7ba2a1367b1a` |
| TabDPT v1.2 Classifier | `kurtvalcorza/tabdpt-classifier-pipeline` | `1e35dc4738f015b254b1724fd4bc7ba9f49668af` | `Layer6/TabDPT` @ `4462ffbd1d8dea25d4862d30beed4b70cd596ae5` |
| TabPFN-3 Classifier | `kurtvalcorza/tabpfn-classifier-pipeline` | `769fb7b47e01b1d10edd954dabc634b8034598c2` | `Prior-Labs/tabpfn_3` @ `24a16a89d245878b846555110985634aa2e656d7` |
| TabICLv2 Classifier | `kurtvalcorza/tabicl-classifier-pipeline` | `7400516ac86e3b6e55f5134ad5e550f312948981` | `jingang/TabICL` @ `4dcd344ece2c00be9e831fdd35bed57b5ad83e19` |

This workshop is a comparative teaching notebook, not a generated carrier for any one pipeline repository. Its results require a clean end-to-end runtime validation before the notebook should be called workshop-ready.
